In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import time
import numpy as np
import pickle
import gc
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import TensorDataset, DataLoader
from concurrent.futures import ThreadPoolExecutor
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from matbench.bench import MatbenchBenchmark
from pymatgen.io.ase import AseAtomsAdaptor
import pandas as pd
from matminer.featurizers.composition import ElementProperty

SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TASK_NAME = "matbench_mp_e_form"
BATCH_SIZE = 32
NUM_THREADS = 8
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
from mace.calculators import mace_mp
from mace.tools.utils import AtomicNumberTable
from mace.data import AtomicData
from mace.data.utils import config_from_atoms, KeySpecification
from mace.tools.torch_geometric.dataloader import DataLoader as MACEDataLoader
from mace.modules.utils import extract_invariant
from e3nn import o3

calc = mace_mp(model="mh-1", default_dtype="float32", device=DEVICE, head="omat_pbe")
model = calc.models[0]
model.eval()

z_table = AtomicNumberTable([int(z) for z in model.atomic_numbers])
r_max = model.r_max.item()
num_interactions = int(model.num_interactions)
irreps_out = o3.Irreps(str(model.products[0].linear.irreps_out))
l_max = irreps_out.lmax
num_invariant_features = irreps_out.dim // (l_max + 1) ** 2
DESC_DIM = num_invariant_features * num_interactions
N_CALC_FEATS = 19

adaptor = AseAtomsAdaptor()
key_spec = KeySpecification()

print(f"Model loaded. Device={DEVICE}, DESC_DIM={DESC_DIM}")

In [ ]:
mb = MatbenchBenchmark(autoload=False, subset=[TASK_NAME])
task = list(mb.tasks)[0]
task.load()
structures = list(task.df["structure"])
n_total = len(structures)
df_index = list(task.df.index)
label_to_pos = {label: pos for pos, label in enumerate(df_index)}
print(f"Loaded {n_total} structures")

In [ ]:
def struct_to_atomic_data(struct):
    atoms = adaptor.get_atoms(struct, msonable=False)
    config = config_from_atoms(atoms, key_specification=key_spec)
    return AtomicData.from_config(
        config, z_table=z_table, cutoff=r_max, heads=["omat_pbe"])


def calc_feats_from_output(output, batch_dict, ptr, i):
    start, end = ptr[i].item(), ptr[i + 1].item()
    n_atoms = end - start
    energy = output["energy"][i].item() if "energy" in output else np.nan
    epa = energy / n_atoms if not np.isnan(energy) else np.nan

    node_e = output.get("node_energy", None)
    if node_e is not None:
        ae = node_e[start:end].detach().cpu().numpy().flatten()
        ae_mean, ae_std = np.mean(ae), np.std(ae)
        ae_min, ae_max = np.min(ae), np.max(ae)
        ae_range = ae_max - ae_min
    else:
        ae_mean = ae_std = ae_min = ae_max = ae_range = np.nan

    forces = output.get("forces", None)
    if forces is not None:
        f = forces[start:end].detach().cpu().numpy()
        fm = np.linalg.norm(f, axis=1)
        fm_mean, fm_std, fm_max = np.mean(fm), np.std(fm), np.max(fm)
    else:
        fm_mean = fm_std = fm_max = np.nan

    stress = output.get("stress", None)
    if stress is not None and stress.numel() > 0:
        s = stress[i].detach().cpu().numpy().flatten()
        if len(s) >= 6:
            hydro = (s[0] + s[1] + s[2]) / 3.0
            max_abs = np.max(np.abs(s[:6]))
            s_feats = list(s[:6]) + [hydro, max_abs]
        else:
            s_feats = [np.nan] * 8
    else:
        s_feats = [np.nan] * 8

    return [energy, epa, n_atoms,
            ae_mean, ae_std, ae_min, ae_max, ae_range,
            fm_mean, fm_std, fm_max] + s_feats

In [ ]:
descs = np.zeros((n_total, DESC_DIM), dtype=np.float32)
calc_feats = np.full((n_total, N_CALC_FEATS), np.nan, dtype=np.float32)
failed = 0
t0 = time.time()
CHUNK = BATCH_SIZE * 4

for chunk_start in range(0, n_total, CHUNK):
    chunk_end = min(chunk_start + CHUNK, n_total)
    chunk_structs = structures[chunk_start:chunk_end]

    valid_data = []
    with ThreadPoolExecutor(max_workers=NUM_THREADS) as executor:
        futures = {executor.submit(struct_to_atomic_data, s): i
                   for i, s in enumerate(chunk_structs)}
        for future in futures:
            local_idx = futures[future]
            global_idx = chunk_start + local_idx
            try:
                data = future.result()
                valid_data.append((global_idx, data))
            except Exception:
                failed += 1

    if not valid_data:
        continue

    valid_data.sort(key=lambda x: x[0])
    indices = [x[0] for x in valid_data]
    data_list = [x[1] for x in valid_data]

    loader = MACEDataLoader(data_list, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

    desc_offset = 0
    for batch in loader:
        batch_size_actual = len(batch.ptr) - 1
        try:
            batch = batch.to(DEVICE)
            batch_dict = batch.to_dict()
            batch_dict["positions"].requires_grad_(True)
            output = model(batch_dict)
            node_feats = output["node_feats"].detach()
            inv_feats = extract_invariant(
                node_feats, num_layers=num_interactions,
                num_features=num_invariant_features, l_max=l_max)
            ptr = batch.ptr
            for i in range(len(ptr) - 1):
                start, end = ptr[i].item(), ptr[i + 1].item()
                descs[indices[desc_offset]] = inv_feats[start:end].mean(dim=0).cpu().numpy()
                try:
                    calc_feats[indices[desc_offset]] = calc_feats_from_output(output, batch_dict, ptr, i)
                except Exception:
                    pass
                desc_offset += 1
        except (torch.cuda.OutOfMemoryError, RuntimeError) as e:
            if "out of memory" not in str(e).lower():
                raise
            torch.cuda.empty_cache()
            for j in range(batch_size_actual):
                try:
                    single = data_list[desc_offset]
                    sl = MACEDataLoader([single], batch_size=1, shuffle=False)
                    sb = next(iter(sl)).to(DEVICE)
                    sd = sb.to_dict()
                    sd["positions"].requires_grad_(True)
                    out = model(sd)
                    nf = out["node_feats"].detach()
                    inv = extract_invariant(nf, num_layers=num_interactions,
                                            num_features=num_invariant_features, l_max=l_max)
                    descs[indices[desc_offset]] = inv.mean(dim=0).cpu().numpy()
                    try:
                        calc_feats[indices[desc_offset]] = calc_feats_from_output(out, sd, sb.ptr, 0)
                    except Exception:
                        pass
                except Exception:
                    failed += 1
                    torch.cuda.empty_cache()
                desc_offset += 1

    done = min(chunk_end, n_total)
    elapsed = time.time() - t0
    rate = done / elapsed
    if done % (CHUNK * 10) < CHUNK:
        print(f"{done}/{n_total} ({rate:.0f}/s, ETA {(n_total-done)/rate:.0f}s, {failed} failures)")

print(f"Extraction done: {n_total} structures, {failed} failures, {time.time()-t0:.0f}s")

In [ ]:
compositions = [s.composition for s in structures]
df_comp = pd.DataFrame({"composition": compositions})
magpie = ElementProperty.from_preset("magpie")
df_comp = magpie.featurize_dataframe(df_comp, "composition", ignore_errors=True)
mm_cols = [c for c in df_comp.columns if c != "composition"]
X_magpie = df_comp[mm_cols].values.astype(np.float32)
print(f"Magpie features: {X_magpie.shape}")

In [ ]:
del structures, model, calc; gc.collect()
torch.cuda.empty_cache()

n_desc = descs.shape[1]
n_calc = calc_feats.shape[1]
n_magpie = X_magpie.shape[1]
n_feat = n_desc + n_calc + n_magpie

X_all = np.empty((n_total, n_feat), dtype=np.float32)
X_all[:, :n_desc] = descs; del descs
X_all[:, n_desc:n_desc+n_calc] = calc_feats; del calc_feats
X_all[:, n_desc+n_calc:] = X_magpie; del X_magpie
gc.collect()

print(f"Combined features: {X_all.shape} ({n_desc} desc + {n_calc} calc + {n_magpie} magpie)")

In [ ]:
class PropertyMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim=256, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.SiLU(),
            nn.LayerNorm(hidden_dim),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1),
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)


def train_mlp_fold(X_train, y_train, X_test, epochs=200, lr=1e-3,
                   batch_size=512, hidden_dim=256, dropout=0.1):
    torch.manual_seed(SEED)
    np.random.seed(SEED)

    tr_idx, val_idx = train_test_split(
        range(len(X_train)), test_size=0.1, random_state=SEED)

    scaler = StandardScaler()
    X_tr_np = scaler.fit_transform(X_train[tr_idx])
    X_val_np = scaler.transform(X_train[val_idx])
    X_te_np = scaler.transform(X_test)

    X_tr = torch.tensor(X_tr_np, dtype=torch.float32)
    y_tr = torch.tensor(y_train[tr_idx], dtype=torch.float32)
    X_val = torch.tensor(X_val_np, dtype=torch.float32)
    y_val = torch.tensor(y_train[val_idx], dtype=torch.float32)
    X_te = torch.tensor(X_te_np, dtype=torch.float32)
    del X_tr_np, X_val_np, X_te_np; gc.collect()

    train_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=batch_size)

    mlp = PropertyMLP(input_dim=X_train.shape[1], hidden_dim=hidden_dim, dropout=dropout).to(DEVICE)
    optimizer = optim.AdamW(mlp.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs)
    loss_fn = nn.L1Loss()

    best_val_mae = float('inf')
    best_state = None

    for epoch in range(epochs):
        mlp.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = loss_fn(mlp(xb), yb)
            loss.backward()
            optimizer.step()
        scheduler.step()

        mlp.eval()
        with torch.no_grad():
            vp = torch.cat([mlp(xb.to(DEVICE)).cpu() for xb, _ in val_loader])
            val_mae = torch.mean(torch.abs(vp - y_val)).item()

        if val_mae < best_val_mae:
            best_val_mae = val_mae
            best_state = {k: v.cpu().clone() for k, v in mlp.state_dict().items()}

    mlp.load_state_dict(best_state)
    mlp.to(DEVICE).eval()
    with torch.no_grad():
        preds = mlp(X_te.to(DEVICE)).cpu().numpy()
    return preds

In [ ]:
mb = MatbenchBenchmark(autoload=False, subset=[TASK_NAME])
task = list(mb.tasks)[0]
task.load()

for fold in task.folds:
    train_inputs, train_outputs = task.get_train_and_val_data(fold)
    test_inputs = task.get_test_data(fold, include_target=False)

    tr_pos = np.array([label_to_pos[l] for l in train_inputs.index])
    te_pos = np.array([label_to_pos[l] for l in test_inputs.index])

    X_tr = X_all[tr_pos]
    X_te = X_all[te_pos]
    y_tr = train_outputs.values

    imp = SimpleImputer(strategy="median")
    X_tr = np.nan_to_num(imp.fit_transform(X_tr), nan=0.0, posinf=0.0, neginf=0.0)
    X_te = np.nan_to_num(imp.transform(X_te), nan=0.0, posinf=0.0, neginf=0.0)

    preds = train_mlp_fold(X_tr, y_tr, X_te)
    task.record(fold, preds)
    print(f"Fold {fold} done")

print(task.scores)

In [ ]:
mb.to_file("results.json.gz")
print("Saved results.json.gz")